In [2]:
import sys
print(sys.executable)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# ── 0. DATEI LADEN ───────────────────────────────────────────
# Pfad anpassen!
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"

cols = ["packaging", "packaging_tags", "packaging_en", "packaging_text"]
df = pd.read_csv(CSV_PATH, sep="	", usecols=cols, low_memory=False)

# Füllrate aller 4 Spalten
print(df[cols].notna().mean().round(2) * 100)

# Alle Paare auf Überlappung prüfen
import itertools

for col1, col2 in itertools.combinations(cols, 2):
    beide = df[[col1, col2]].notna().all(axis=1).sum()
    print(f"{col1} + {col2}: beide befüllt in {beide} Zeilen")

# Paarweise Überlappung aller 4 Spalten (wie oft sind je 2 gleichzeitig befüllt?)
overlap = pd.DataFrame(index=cols, columns=cols, dtype=float)
for c1 in cols:
    for c2 in cols:
        overlap.loc[c1, c2] = df[[c1, c2]].notna().all(axis=1).sum()
overlap.astype(int)

/opt/homebrew/opt/python@3.11/bin/python3.11
packaging         8.0
packaging_tags    8.0
packaging_en      8.0
packaging_text    1.0
dtype: float64
packaging + packaging_tags: beide befüllt in 378933 Zeilen
packaging + packaging_en: beide befüllt in 378929 Zeilen
packaging + packaging_text: beide befüllt in 18916 Zeilen
packaging_tags + packaging_en: beide befüllt in 378934 Zeilen
packaging_tags + packaging_text: beide befüllt in 18915 Zeilen
packaging_en + packaging_text: beide befüllt in 18915 Zeilen


,packaging,packaging_tags,packaging_en,packaging_text
packaging,378953,378933,378929,18916
packaging_tags,378933,378945,378934,18915
packaging_en,378929,378934,378934,18915
packaging_text,18916,18915,18915,31777


In [ ]:
# Zeilen wo mehrere Spalten gleichzeitig befüllt sind
mask = df[cols].notna().sum(axis=1) >= 2
df[cols][mask].head(20)

,packaging,packaging_tags,packaging_en,packaging_text
15,"Plastic,Cardboard,fr:Boîte en carton,fr:Film e...","en:plastic,en:cardboard,fr:boite-en-carton,fr:...","Plastic,Cardboard,fr:boite-en-carton,fr:film-e...",NaN
18,pot plastique,pot-plastique,Pot-plastique,NaN
19,"Plastique, Carton","en:plastic,en:cardboard","Plastic,Cardboard",NaN
24,1 boîte en carton à recycler 50 sachets indivi...,fr:1-boite-en-carton-a-recycler-50-sachets-ind...,fr:1-boite-en-carton-a-recycler-50-sachets-ind...,NaN
26,"Boîte en carton, Film en plastique","fr:boite-en-carton,fr:film-en-plastique","fr:boite-en-carton,fr:film-en-plastique",NaN
28,Packung(en),de:packung-en,de:packung-en,NaN
32,Plastic,en:plastic,Plastic,NaN
37,Plastique,en:plastic,Plastic,NaN
38,Plastique,en:plastic,Plastic,NaN
45,Plastique,en:plastic,Plastic,NaN


In [6]:
# Nimm packaging_en, falls leer dann packaging als Fallback
# Reihenfolge = Priorität (erste Spalte hat Vorrang)
df['packaging_combined'] = (
    df['packaging_en']
    .fillna(df['packaging'])
    .fillna(df['packaging_text'])
    .fillna(df['packaging_tags'])
)

In [7]:

# Als Parquet speichern


df.to_parquet('packaging_bereinigung.parquet', index=False)